# 01. The linkage procedure on a toy example

This notebook explains the procedure used to link public, de-identified live birth records (SINASC) to neonatal death
records (SIM) in Brazil without a unique identifier. It runs on a **small invented example**, so nothing here needs the
public microdata. The real-data results are in `02_results.ipynb`, and the simulation-based evaluation is in
`03_synthetic_evaluation.ipynb`.

Stages of the procedure (v4.2):

1. **Exact passes** P1-P3 over an index of the run year's births plus the previous December's births.
2. **Tolerant candidates** for deaths left unmatched (date ±1 day or swapped digits, weight ±10 g or rounded, mother's age ±1).
3. **Pair scores** from a classifier on seven comparison inputs.
4. **One-to-one assignment** (Hungarian algorithm), accepting pairs with score at or above τ.
5. **Consolidation** of overlapping annual runs into one base with record identities.

The linkage yield is the proportion of death records that receive an accepted birth record. It does not measure whether the
accepted pairs are correct.

In [1]:
import numpy as np, pandas as pd
from datetime import datetime, timedelta
from scipy.optimize import linear_sum_assignment
pd.set_option("display.width", 140)

## Toy data

Births carry date, sex, municipality, facility, weight and mother's age. Deaths carry the same fields with a few discrepancies and missing values.

In [2]:
births = pd.DataFrame([
    # id, date (ddmmyyyy), sex, municipality, facility, weight, mother_age
    (0, "05032019", "1", "350010", "2001", "3100", "27"),
    (1, "05032019", "2", "350010", "2001", "2850", "31"),
    (2, "06032019", "1", "350020", "2002", "1500", "22"),
    (3, "07032019", "2", "350010", "2003", "2480", "19"),
    (4, "08032019", "1", "350030", "2004", "3400", "35"),
    (5, "09032019", "2", "350020", "2002", "2900", "28"),
    (6, "10032019", "1", "350010", "2001", "1200", "24"),
    (7, "11032019", "2", "350030", "2004", "3050", "30"),
], columns=["birth_id", "date", "sex", "mun", "facility", "weight", "age"])

deaths = pd.DataFrame([
    # d0: exact match with birth 0
    (0, "05032019", "1", "350010", "2001", "3100", "27"),
    # d1: weight recorded rounded to 50 g (2900 instead of 2850): no exact key, found through weight variants
    (1, "05032019", "2", "350010", "2001", "2900", "31"),
    # d2: date recorded one day late (07 instead of 06): no exact key, found through date variants
    (2, "07032019", "1", "350020", "2002", "1500", "22"),
    # d3: weight missing; age present
    (3, "07032019", "2", "350010", "2003", None, "19"),
    # d4: no true birth in the toy index
    (4, "20032019", "1", "350030", "2004", "2200", "40"),
    # d5: exact match with birth 5
    (5, "09032019", "2", "350020", "2002", "2900", "28"),
], columns=["death_id", "date", "sex", "mun", "facility", "weight", "age"])
truth = {0: 0, 1: 1, 2: 2, 3: 3, 4: None, 5: 5}
births, deaths

(   birth_id      date sex     mun facility weight age
 0         0  05032019   1  350010     2001   3100  27
 1         1  05032019   2  350010     2001   2850  31
 2         2  06032019   1  350020     2002   1500  22
 3         3  07032019   2  350010     2003   2480  19
 4         4  08032019   1  350030     2004   3400  35
 5         5  09032019   2  350020     2002   2900  28
 6         6  10032019   1  350010     2001   1200  24
 7         7  11032019   2  350030     2004   3050  30,
    death_id      date sex     mun facility weight age
 0         0  05032019   1  350010     2001   3100  27
 1         1  05032019   2  350010     2001   2900  31
 2         2  07032019   1  350020     2002   1500  22
 3         3  07032019   2  350010     2003    NaN  19
 4         4  20032019   1  350030     2004   2200  40
 5         5  09032019   2  350020     2002   2900  28)

## Stage 1. Exact passes

A pair needs every key field present. A death without weight cannot link in P1-P3.

In [3]:
def exact_passes(births, deaths):
    keys = {
        "P1": lambda r: (r.date, r.sex, r.mun, r.weight),
        "P2": lambda r: (r.date, r.sex, r.mun, r.weight),     # municipality of occurrence stands in for the second municipality field here
        "P3": lambda r: (r.date, r.facility, r.weight),
    }
    used, matches = set(), {}
    for p, f in keys.items():
        idx = {}
        for r in births.itertuples():
            k = f(r)
            if not any(pd.isna(x) for x in k):
                idx.setdefault(k, []).append(r.birth_id)
        for r in deaths.itertuples():
            if r.death_id in matches:
                continue
            k = f(r)
            if any(pd.isna(x) for x in k):
                continue
            for b in idx.get(k, []):
                if b not in used:
                    used.add(b); matches[r.death_id] = (b, p); break
    return matches, used

matches, used = exact_passes(births, deaths)
matches

{0: (0, 'P1'), 5: (5, 'P1')}

## Stage 2. Tolerant candidates

For each death left unmatched, births are found by three keys over variants. Weight variants are the recorded weight,
±10 g, and the weight rounded to the nearest 50 and 100 g. Date variants are the recorded date, one day before or after,
and the date with its first two digits swapped when valid. Age variants are the recorded age and ±1 year.
The third key (date, sex, municipality, mother's age) does not need weight.

In [4]:
def date_variants(dt):
    if pd.isna(dt) or not dt: return []
    out = [dt]
    try:
        d = datetime.strptime(dt, "%d%m%Y")
    except ValueError:
        return out
    out += [(d + timedelta(days=1)).strftime("%d%m%Y"), (d - timedelta(days=1)).strftime("%d%m%Y")]
    sw = dt[1] + dt[0] + dt[2:]
    if sw != dt:
        try:
            datetime.strptime(sw, "%d%m%Y"); out.append(sw)
        except ValueError:
            pass
    return out

def weight_variants(w):
    if pd.isna(w): return []
    w = int(float(w))
    return list({str(w), str(w + 10), str(w - 10), str(int(round(w / 50) * 50)), str(int(round(w / 100) * 100))})

def age_variants(a):
    if pd.isna(a): return []
    a = int(float(a)); return [str(a), str(a + 1), str(a - 1)]

def candidates(death, births, used):
    out = set()
    for r in births.itertuples():
        if r.birth_id in used: continue
        if r.date in date_variants(death.date):
            if r.sex == death.sex and r.mun == death.mun and r.weight in weight_variants(death.weight):
                out.add(r.birth_id)
            if r.facility == death.facility and r.weight in weight_variants(death.weight):
                out.add(r.birth_id)
            if r.sex == death.sex and r.mun == death.mun and r.age in age_variants(death.age):
                out.add(r.birth_id)
    return out

cand = {d.death_id: candidates(d, births, used) for d in deaths.itertuples() if d.death_id not in matches}
cand

{1: {1}, 2: {2}, 3: {3}, 4: set()}

## Stage 3. Comparison inputs and scores

Seven inputs describe each death-candidate pair: date state, sex state, municipality state, weight state, mother's age
state, facility state, and the number of candidates (capped at 50). The state codes are documented in the repository
README. In the study a gradient-boosted classifier trained once on a simulated reference scored each pair.
Here, to keep the example self-contained, a **transparent hand-set score** replaces the classifier. It is an illustration,
not the trained model.

In [5]:
def date_state(a, b):
    if pd.isna(a) or pd.isna(b): return 3
    if a == b: return 0
    try:
        if abs((datetime.strptime(a, "%d%m%Y") - datetime.strptime(b, "%d%m%Y")).days) == 1: return 1
    except ValueError:
        return 3
    return 2 if a[1] + a[0] + a[2:] == b else 3

def weight_state(a, b):
    if pd.isna(a) or pd.isna(b): return 4
    wa, wb = int(float(a)), int(float(b)); d = abs(wa - wb)
    if d == 0: return 0
    if d <= 10: return 1
    if wa == round(wb / 50) * 50 or wb == round(wa / 50) * 50: return 2
    if wa == round(wb / 100) * 100 or wb == round(wa / 100) * 100: return 3
    return 5

def age_state(a, b):
    if pd.isna(a) or pd.isna(b): return 2
    d = abs(int(a) - int(b)); return 0 if d == 0 else (1 if d == 1 else 3)

def features(death, birth, n):
    sex = 0 if death.sex == birth.sex else 2
    mun = 0 if death.mun == birth.mun else 2
    fac = 2 if (pd.isna(death.facility) or pd.isna(birth.facility)) else (0 if death.facility == birth.facility else 1)
    return [date_state(death.date, birth.date), sex, mun, weight_state(death.weight, birth.weight),
            age_state(death.age, birth.age), fac, min(n, 50)]

def toy_score(f):
    # hand-set penalties, for illustration only
    pen = {0: 0, 1: .15, 2: .2, 3: .5, 4: .25, 5: .6}
    s = 1.0 - .12 * f[0] - .25 * (f[1] == 2) - .25 * (f[2] == 2) - pen[f[3]] * .5 - .1 * (f[4] == 3) - .05 * (f[5] == 1)
    return max(s, 0.0)

rows = []
for did, bs in cand.items():
    d = next(deaths[deaths.death_id == did].itertuples())
    for b in bs:
        br = next(births[births.birth_id == b].itertuples())
        f = features(d, br, len(bs))
        rows.append(dict(death=did, birth=b, features=f, score=round(toy_score(f), 3)))
pairs = pd.DataFrame(rows); pairs

,death,birth,features,score
0,1,1,"[0, 0, 0, 5, 0, 0, 1]",0.700
1,2,2,"[1, 0, 0, 0, 0, 0, 1]",0.880
2,3,3,"[0, 0, 0, 4, 0, 0, 1]",0.875


## Stage 4. One-to-one assignment

The Hungarian algorithm maximizes the total score within each block of competing candidates; pairs below τ are rejected. The assignment is decided before τ is applied.

In [6]:
def assign(pairs, tau):
    if pairs.empty: return {}
    ds, bs = sorted(pairs.death.unique()), sorted(pairs.birth.unique())
    cost = np.full((len(ds), len(bs)), 1e6)
    for r in pairs.itertuples():
        cost[ds.index(r.death), bs.index(r.birth)] = -r.score
    ri, ci = linear_sum_assignment(cost)
    return {ds[i]: (bs[j], -cost[i, j]) for i, j in zip(ri, ci) if -cost[i, j] >= tau}

tau = 0.3
clf = assign(pairs, tau)
final = {d: (b, "exact " + p) for d, (b, p) in matches.items()}
final.update({d: (b, f"classifier {s:.2f}") for d, (b, s) in clf.items()})
res = pd.DataFrame([(d, *final.get(d, (None, "unlinked"))) for d in deaths.death_id], columns=["death", "assigned_birth", "stage"])
res["true_birth_in_toy_data"] = res.death.map(truth)
res

,death,assigned_birth,stage,true_birth_in_toy_data
0,0,0.0,exact P1,0.0
1,1,1.0,classifier 0.70,1.0
2,2,2.0,classifier 0.88,2.0
3,3,3.0,classifier 0.88,3.0
4,4,NaN,unlinked,NaN
5,5,5.0,exact P1,5.0


In the toy data, death 4 has no birth in the index and stays unlinked. In real data, an unlinked death can also come from
a missing key field (for example a missing weight), an unregistered birth, or a discrepancy larger than the tolerances.
The **toy truth column is known only because the example is invented**. For the public files no independent reference of
true pairs was available for this study.

## Stage 5. Consolidation across overlapping annual runs

Deaths of January in year y+1 born in year y are proposed in the run of year y and again in the run of y+1 (against
December births). Every proposed pair is recorded with identities of both records and consolidated by a fixed rule:
repeated pairs are collapsed, pairs are ordered by stage, score, own-file run before January-window run, and run year,
and a pair is accepted only if neither record is already assigned.

In [7]:
proposals = pd.DataFrame([
    # run_year, death (file, row), birth (file, row), stage rank (1-3 exact, 4 classifier), score, january-window run?
    (2019, ("sim2020", 11), ("sinasc2019", 501), 1, np.nan, 1),
    (2020, ("sim2020", 11), ("sinasc2019", 501), 1, np.nan, 0),   # same pair proposed again by the next run
    (2019, ("sim2020", 12), ("sinasc2019", 777), 4, 0.62, 1),
    (2020, ("sim2020", 13), ("sinasc2019", 777), 4, 0.48, 0),      # the same birth proposed to another death
], columns=["run_year", "death", "birth", "stage_rank", "score", "jan_run"])
proposals["neg_score"] = -proposals.score.fillna(0)
order = proposals.sort_values(["stage_rank", "neg_score", "jan_run", "run_year"])
taken_d, taken_b, kept = set(), set(), []
for r in order.drop_duplicates(["death", "birth"]).itertuples():
    if r.death in taken_d or r.birth in taken_b:
        continue
    taken_d.add(r.death); taken_b.add(r.birth); kept.append(r)
pd.DataFrame(kept)[["run_year", "death", "birth", "stage_rank", "score"]]

,run_year,death,birth,stage_rank,score
0,2020,"(sim2020, 11)","(sinasc2019, 501)",1,NaN
1,2019,"(sim2020, 12)","(sinasc2019, 777)",4,0.62


The consolidated base has at most one birth per death and at most one death per birth. The rule is operational and does
not show that the prioritized pair is correct.

## Frozen classifier (optional)

If `xgboost` is installed, the trained classifier file in `model/` can score comparison vectors. Its hyperparameters come
from the training code (300 trees, depth 5, learning rate 0.1, subsample and column sample 0.9, seed 42), and the file
itself stores 300 rounds with a maximum observed tree depth of 5.

In [8]:
try:
    import xgboost as xgb, json, os
    m = xgb.XGBClassifier(); m.load_model(os.path.join("model", "xgb_pairs.json"))
    toy = np.array([[0, 0, 0, 0, 0, 0, 1], [1, 0, 0, 2, 0, 0, 2], [3, 2, 2, 5, 3, 1, 5]], dtype=float)
    print("rounds:", m.get_booster().num_boosted_rounds())
    print("scores of three illustrative comparison vectors:", np.round(m.predict_proba(toy)[:, 1], 3))
except Exception as e:
    print("xgboost or the model file is not available here:", type(e).__name__)

xgboost or the model file is not available here: ModuleNotFoundError
